# z629 - Regresion Lineal: 2 anos, MISMA transicion estacional (dic->feb)

## Que aisla este notebook
`z625` (original, 0.231) entrena solo con `periodo=201812`, target=tn de `201902` (transicion dic->feb, la misma que hay que predecir para 202002). `z626` (0.278, peor) uso TODOS los meses, mezclando transiciones de otras estaciones.

Esta version prueba una hipotesis intermedia: mantener la transicion estacional dic->feb (no mezclar estaciones), pero usar 2 anos de esa transicion en vez de 1 (`Dec2017->Feb2018` + `Dec2018->Feb2019`). Mismos productos magicos, mismo modelo OLS.

Si esto mejora sobre 0.231, la lectura es "mas data ayuda, siempre que sea la misma estacion". Si no mejora, un solo anio (el mas reciente) ya era suficiente y mas data no aporta.

In [1]:
!pip install -q polars statsmodels

In [2]:
import os
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'LR03_2ANIOS',
    'kaggle_competition': 'labo-iii-2026-ba',
    'datasets_path': '/home/ds/datasets/',
    'exp_path': '/home/ds/exp/',
    'periodos_entrenamiento': [201712, 201812]   # ambos diciembre -- misma transicion dic->feb
}

ruta = os.path.join(PARAM['exp_path'], PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LR03_2ANIOS


## 1. Cargar y filtrar a productos a predecir (identico a z625)

In [4]:
dataset = pl.read_csv(os.path.join(PARAM['datasets_path'], 'sell-in.txt.gz'), separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv(os.path.join(PARAM['datasets_path'], 'product_id_apredecir201912.txt'), separator="\t")

print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner")
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


## 2. Lags y clase (identico a z625: 12 lags, target a 2 periodos)

In [5]:
lags = [-2, *range(0, 12)]

tb_lags = (
    tb_ventas.sort(["product_id", "periodo"])
    .with_columns(
        [
            pl.col("tn").shift(lag).over("product_id").alias(f"tn_{lag}")
            for lag in lags
        ]
    )
)
tb_lags = tb_lags.rename({"tn_-2": "clase"})

## 3. Productos magicos (misma lista que z625)

In [6]:
productos_magicos = [ 20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
  20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
  20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
  20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
  20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
  20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
  20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
  20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
  20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
  20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
  20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
  20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
  20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
  20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
  20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
  21080, 21088, 21118, 21170, 21200
]

## 4. Entrenar con AMBOS diciembres (2017 y 2018), mismos productos magicos

In [7]:
campos_lag = [f"tn_{n}" for n in range(0, 12)]

dtrain = tb_lags.filter(
    pl.col("periodo").is_in(PARAM['periodos_entrenamiento']) &
    pl.col("product_id").is_in(productos_magicos) &
    pl.all_horizontal([pl.col(c).is_not_null() for c in campos_lag]) &
    pl.col("clase").is_not_null()
)
print("filas de entrenamiento (2 anios x productos magicos, con historial completo):", dtrain.shape)
print(dtrain.group_by("periodo").len())

filas de entrenamiento (2 anios x productos magicos, con historial completo): (336, 16)
shape: (2, 2)
┌─────────┬─────┐
│ periodo ┆ len │
│ ---     ┆ --- │
│ i64     ┆ u32 │
╞═════════╪═════╡
│ 201812  ┆ 182 │
│ 201712  ┆ 154 │
└─────────┴─────┘


## 5. Modelo OLS (identico a z625)

In [8]:
campos_buenos = dtrain.select(cs.starts_with("tn_"))

X_train = dtrain.select(campos_buenos).to_pandas()
X_train = sm.add_constant(X_train)
y_train = dtrain['clase'].to_pandas()

modelo = sm.OLS(y_train, X_train).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  clase   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     2042.
Date:                Sat, 15 Aug 2026   Prob (F-statistic):          2.71e-296
Time:                        15:29:21   Log-Likelihood:                -1366.5
No. Observations:                 336   AIC:                             2759.
Df Residuals:                     323   BIC:                             2809.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4404      0.938     -0.469      0.6

## 6. Aplicacion a 201912 (identico a z625: solo productos con historial completo, fallback a promedio)

In [9]:
dfuture = tb_lags.filter( (pl.col("periodo") == 201912) & (pl.col("tn_11").is_not_null()) )
print(dfuture.shape)

(656, 16)


In [10]:
campos_buenos = dfuture.select(cs.starts_with("tn_"))
X_future = dfuture.select(campos_buenos).to_pandas()
X_future = sm.add_constant(X_future, has_constant='add')

prediccion = modelo.predict(X_future)

tb_regresion = dfuture.select(['product_id']).with_columns(
    pl.Series("tn_pred", prediccion)
)
print(tb_regresion.shape)

(656, 2)


In [11]:
primer_periodo = 201901
ultimo_periodo = 201912
tb_meses12 = tb_ventas.filter(pl.col("periodo").is_between(primer_periodo, ultimo_periodo)).group_by("product_id").agg(
    pl.col("tn").mean().alias("tn")
)
tb_meses12 = tb_meses12.select(["product_id", "tn"])

tb_final = (
    tb_meses12
    .join(tb_regresion, on="product_id", how="left", suffix="_update")
    .with_columns(
        pl.coalesce([pl.col("tn_pred"), pl.col("tn")]).alias("tn")
    )
    .drop("tn_pred")
)
print(tb_final.shape)
tb_final.head()

(780, 2)


product_id,tn
i64,f64
20001,1221.197654
20002,1033.617512
20003,769.508097
20004,519.943957
20005,480.617664


## 7. Submit

In [12]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

archivo = os.path.join(ruta, "linreg_2anios.csv")
tb_final.write_csv(archivo)
print(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, "Regresion Lineal - 2 anios, misma estacion dic-feb")

/home/ds/exp/LR03_2ANIOS/linreg_2anios.csv


100%|██████████| 18.6k/18.6k [00:00<00:00, 56.9kB/s]


92 submissions remaining today.
Successfully submitted to Labo III, 2026 BA